# House Prices using Backward Elimination

Just started with machine learning. I have used backward Elimination to check the usefulness of dependent variables.

In [2]:
# Importing Libraries
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

In [3]:
# Loading dataset
Housing = pd.read_csv(r'C:\Users\akash\OneDrive\Desktop\FSDS\ML\MLR\MLR\House_data.csv')
Housing.head()

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


In [6]:
# Checking Missing Values
print(Housing.isnull().any())

id               False
date             False
price            False
bedrooms         False
bathrooms        False
sqft_living      False
sqft_lot         False
floors           False
waterfront       False
view             False
condition        False
grade            False
sqft_above       False
sqft_basement    False
yr_built         False
yr_renovated     False
zipcode          False
lat              False
long             False
sqft_living15    False
sqft_lot15       False
dtype: bool


In [7]:
# Checking Categorical data
print(Housing.dtypes)

id                 int64
date              object
price            float64
bedrooms           int64
bathrooms        float64
sqft_living        int64
sqft_lot           int64
floors           float64
waterfront         int64
view               int64
condition          int64
grade              int64
sqft_above         int64
sqft_basement      int64
yr_built           int64
yr_renovated       int64
zipcode            int64
lat              float64
long             float64
sqft_living15      int64
sqft_lot15         int64
dtype: object


In [8]:
# Droping id and date column
Housing = Housing.drop(['id', 'date'], axis = 1)

In [11]:
# Understand the distribution with seaborn
with sns.plotting_context('notebook', font_scale=2.5):
    g = sns.pairplot(
        Housing[['sqft_lot', 'sqft_above', 'price', 'sqft_living', 'bedrooms']],
        hue = 'bedrooms', palette = 'tab20', height = 6
    )
g.set(xticklabels=[]);

In [12]:
# Separating Independent & Dependent Variable
X = Housing.iloc[:,1:].values
y = Housing.iloc[:, 0].values

# Splitting Dataset into training & testing 
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=1/3, random_state=0
)

In [13]:
from sklearn.linear_model import LinearRegression
regression = LinearRegression()
regression.fit(X_train, y_train)

# Predicting the Test results
y_pred = regression.predict(X_test) 

In [18]:
# Backward Elimination
import statsmodels.api as sm
X = sm.add_constant(X)

def backwardElimination(x, SL):
    numVars = len(x[0])
    temp = np.zeros((21613, 19)).astype(int)
    for i in range(0, numVars):
        regression_OLS = sm.OLS(y, x).fit()
        maxVar = max(regression_OLS.pvalues).astype(float)
        adjR_before = regression_OLS.rsquared_adj.astype(float)

        if maxVar > SL:
            for j in range(0, numVars - i):
                if (regression_OLS.pvalues[j].astype(float) == maxVar):
                    temp[:, j] = x[:, j]
                    x = np.delete(x, j, 1)
                    tmp_regression  = sm.OLS(y, x).fit()
                    adjR_after = tmp_regression.rsquared_adj.astype(float)
                    if (adjR_before >= adjR_after):
                        x_rollback = np.hstack((x, temp[:, [0, j]]))
                        x_rollback = np.delete(x_rollback,  j, i)
                        print(regression_OLS.summary())
                        return x_rollback
                    else:
                        continue
    regression_OLS.summary()
    return x


SL = 0.05
X_opt = X[:, [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]]
X_Modeled = backwardElimination(X_opt, SL)

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.699
Model:                            OLS   Adj. R-squared:                  0.699
Method:                 Least Squares   F-statistic:                     3140.
Date:                Wed, 17 Dec 2025   Prob (F-statistic):               0.00
Time:                        22:07:59   Log-Likelihood:            -2.9462e+05
No. Observations:               21613   AIC:                         5.893e+05
Df Residuals:                   21596   BIC:                         5.894e+05
Df Model:                          16                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       6.092e+06   2.93e+06      2.079      0.0

In [19]:
import statsmodels.api as sm

# add constant
X = sm.add_constant(X)

def backwardElimination(x, SL):
    numVars = x.shape[1]

    while True:
        regressor_OLS = sm.OLS(y, x).fit()
        pvalues = regressor_OLS.pvalues

        # exclude constant (index 0)
        max_pval = max(pvalues[1:])

        if max_pval > SL:
            max_index = np.argmax(pvalues[1:]) + 1
            x = np.delete(x, max_index, axis=1)
        else:
            break

    print(regressor_OLS.summary())
    return x


SL = 0.05
X_opt = X
X_Modeled = backwardElimination(X_opt, SL)

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.700
Model:                            OLS   Adj. R-squared:                  0.699
Method:                 Least Squares   F-statistic:                     3145.
Date:                Wed, 17 Dec 2025   Prob (F-statistic):               0.00
Time:                        22:10:32   Log-Likelihood:            -2.9460e+05
No. Observations:               21613   AIC:                         5.892e+05
Df Residuals:                   21596   BIC:                         5.894e+05
Df Model:                          16                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       5.741e+06   2.89e+06      1.989      0.0